In [ ]:
!pip install pandas numpy sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import re

In [ ]:
df = pd.read_csv('/content/data.csv')

In [ ]:
texts = df.iloc[:,0].dropna().astype(str).tolist()

print("Raw Sample:", texts[:3])

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)  # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text

texts_clean = [clean_text(t) for t in texts]

print("Clean Sample:", texts_clean[:3])


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(texts_clean)

print("Embedding Shape:", embeddings.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

In [ ]:
SEQ_LEN = 10   # sequence length (tunable)

def create_3d_matrix(data, seq_len):
    sequences = []

    for i in range(len(data) - seq_len):
        seq = data[i:i+seq_len]
        sequences.append(seq)

    return np.array(sequences)

matrix_3d = create_3d_matrix(embeddings_scaled, SEQ_LEN)

print("Final 3D Matrix Shape:", matrix_3d.shape)

In [ ]:
np.save("text_3d_matrix.npy", matrix_3d)

print("✅ 3D Matrix Saved Successfully!")

In [ ]:
# ==============================
# IMPORTS
# ==============================
import numpy as np
import pandas as pd

# Load your 3D matrix
matrix_3d = np.load("text_3d_matrix.npy")

results = []

# ==============================
# METRIC COMPUTATION
# ==============================
for i in range(len(matrix_3d)):

    mat = matrix_3d[i]

    # Matrix multiplication
    result = np.dot(mat, mat.T)

    # ---------- METRICS ----------

    # Accuracy (stability-based)
    acc = np.mean(result) / (np.std(result) + 1e-5)
    acc = np.tanh(acc) * 100   # normalize to %

    # Cleanliness
    clean = (np.sum(mat != 0) / mat.size) * 100

    # Errors
    errors = (np.sum(np.isnan(mat)) / mat.size) * 100

    # Anomaly (outliers)
    anomaly = (np.sum(np.abs(mat) > 3) / mat.size) * 100

    # Reward (custom weighted %)
    reward = acc + clean - errors - anomaly
    reward = max(0, min(100, reward))  # clamp 0–100

    results.append([acc, clean, errors, anomaly, reward])

# ==============================
# CREATE TABLE
# ==============================
columns = ["Accuracy (%)", "Cleanliness (%)", "Error (%)", "Anomaly (%)", "Reward (%)"]

df_results = pd.DataFrame(results, columns=columns)

# Round values
df_results = df_results.round(2)

# ==============================
# SHOW OUTPUT
# ==============================
print(df_results.head())

# Save
df_results.to_csv("final_results_percentage.csv", index=False)

print("\n✅ Results saved as final_results_percentage.csv")

In [ ]:
!pip install transformers sentence-transformers accelerate


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import pandas as pd
df = pd.read_csv('/content/sample_data/california_housing_train.csv')
texts = df.iloc[:,0].dropna().astype(str).tolist()

In [ ]:
import re

# Ensure 'texts' is defined from 'df' (if not already from previous cell)
# This line is added to handle NameError if previous cell's 'texts' is lost.
texts = df.iloc[:,0].dropna().astype(str).tolist()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text.strip()

texts = [clean_text(t) for t in texts]

In [ ]:
# Define get_embeddings function
# This cell was executed to define the function, which was missing in the previous run.
def get_embeddings(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)

    embeddings = []

    for text in texts[:500]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # ✅ FIXED LINE
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().to(torch.float32).cpu().numpy()

        embeddings.append(emb)

    return np.array(embeddings)

# Now, re-run the cell that calls get_embeddings
print("Loading Qwen...")
qwen_emb = get_embeddings("Qwen/Qwen2-0.5B")

print("Loading LLaMA...")
llama_emb = get_embeddings("sentence-transformers/all-mpnet-base-v2")

In [ ]:
def get_embeddings(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)

    embeddings = []

    for text in texts[:500]:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # ✅ FIXED LINE
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().to(torch.float32).cpu().numpy()

        embeddings.append(emb)

    return np.array(embeddings)

In [ ]:
# This cell was part of the previous 'get_embeddings' function and is now empty. The code has been moved to the preceding cell.

In [ ]:
print("Loading Qwen...")
qwen_emb = get_embeddings("Qwen/Qwen2-0.5B")

print("Loading LLaMA...")
llama_emb = get_embeddings("sentence-transformers/all-mpnet-base-v2")

In [ ]:
scaler = StandardScaler()

qwen_emb = scaler.fit_transform(qwen_emb)
llama_emb = scaler.fit_transform(llama_emb)

In [ ]:
SEQ_LEN = 10

def create_3d(data):
    return np.array([data[i:i+SEQ_LEN] for i in range(len(data)-SEQ_LEN)])

qwen_3d = create_3d(qwen_emb)
llama_3d = create_3d(llama_emb)

In [ ]:
import time

def compute_metrics(matrix_3d):

    results = []
    times = []

    for mat in matrix_3d:

        # ⏱️ START TIMER
        start_time = time.time()

        # Matrix multiplication
        result = np.dot(mat, mat.T)

        # ⏱️ END TIMER
        end_time = time.time()

        exec_time = (end_time - start_time) * 1000  # ms
        times.append(exec_time)

        # ---------- METRICS ----------
        acc = np.mean(result) / (np.std(result) + 1e-5)
        acc = np.tanh(acc) * 100

        clean = (np.sum(mat != 0) / mat.size) * 100
        errors = (np.sum(np.isnan(mat)) / mat.size) * 100
        anomaly = (np.sum(np.abs(mat) > 3) / mat.size) * 100

        reward = acc + clean - errors - anomaly
        reward = max(0, min(100, reward))

        results.append([acc, clean, errors, anomaly, reward, exec_time])

    return np.array(results), np.array(times)

In [ ]:
qwen_results, qwen_time = compute_metrics(qwen_3d)
llama_results, llama_time = compute_metrics(llama_3d)

In [ ]:
qwen_avg = qwen_results.mean(axis=0)
llama_avg = llama_results.mean(axis=0)

In [ ]:
def normalize_time(time_array):
    max_t = np.max(time_array)
    min_t = np.min(time_array)

    # Invert: lower time → higher score
    norm = 100 * (1 - (time_array - min_t) / (max_t - min_t + 1e-5))
    return norm

In [ ]:
qwen_time_score = normalize_time(qwen_time)
llama_time_score = normalize_time(llama_time)

In [ ]:
columns = ["Accuracy (%)", "Cleanliness (%)", "Error (%)", "Anomaly (%)", "Reward (%)", "Time Score (%)"]

qwen_avg = np.hstack([qwen_results.mean(axis=0)[:5], qwen_time_score.mean()])
llama_avg = np.hstack([llama_results.mean(axis=0)[:5], llama_time_score.mean()])

comparison_df = pd.DataFrame(
    [qwen_avg, llama_avg],
    columns=columns,
    index=["Qwen", "LLaMA (proxy)"]
)

comparison_df = comparison_df.round(2)

print("\n🔥 FINAL TABLE WITH OPTIMIZATION TIME:\n")
print(comparison_df)